### Load the preprocessed data

In [1]:
import pandas as pd

X_train = pd.read_csv('dataset/X_train.csv')
y_train = pd.read_csv('dataset/y_train.csv').squeeze()

X_test = pd.read_csv('dataset/X_test.csv')
y_test = pd.read_csv('dataset/y_test.csv').squeeze()

X_train_balanced = pd.read_csv('dataset/X_train_balanced.csv')
y_train_balanced = pd.read_csv('dataset/y_train_balanced.csv').squeeze()

print(
    "Raw train:", X_train.shape,
    "| Balanced train:", X_train_balanced.shape,
    "| Test:", X_test.shape
)

Raw train: (25217, 47) | Balanced train: (39544, 47) | Test: (6305, 47)


### Build a shared results table

In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []


def evaluate_and_log(name, model, X_tr, y_tr, X_te, y_te, cv):

    cv_auc = cross_val_score(
        model,
        X_tr,
        y_tr,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    row = {
        'model': name,
        'cv_roc_auc_mean': round(cv_auc.mean(), 3),
        'test_accuracy': round(accuracy_score(y_te, y_pred), 3),
        'test_precision': round(precision_score(y_te, y_pred), 3),
        'test_recall': round(recall_score(y_te, y_pred), 3),
        'test_f1': round(f1_score(y_te, y_pred), 3),
        'test_roc_auc': round(roc_auc_score(y_te, y_proba), 3)
    }

    results.append(row)

    print(row)

    return model

### Train the Random Forest baseline

In [3]:
from sklearn.ensemble import RandomForestClassifier

rf_baseline = evaluate_and_log(
    'Random Forest (baseline, raw)',
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    X_train,
    y_train,
    X_test,
    y_test,
    cv
)

{'model': 'Random Forest (baseline, raw)', 'cv_roc_auc_mean': np.float64(0.898), 'test_accuracy': 0.897, 'test_precision': 0.939, 'test_recall': 0.558, 'test_f1': 0.7, 'test_roc_auc': 0.9}


### Train the same model on SMOTE-balanced data

In [4]:
rf_balanced = evaluate_and_log(
    'Random Forest (baseline, SMOTE-balanced)',
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    X_train_balanced,
    y_train_balanced,
    X_test,
    y_test,
    cv
)

{'model': 'Random Forest (baseline, SMOTE-balanced)', 'cv_roc_auc_mean': np.float64(0.978), 'test_accuracy': 0.884, 'test_precision': 0.79, 'test_recall': 0.632, 'test_f1': 0.702, 'test_roc_auc': 0.893}


### Hyperparameter tuning

In [5]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_dist,
    n_iter=20,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42
)

rf_search.fit(X_train, y_train)

print("Best params:", rf_search.best_params_)
print("Best CV ROC-AUC:", round(rf_search.best_score_, 3))

Best params: {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
Best CV ROC-AUC: 0.902


### Best parameters and tuned model

In [6]:
rf_tuned = evaluate_and_log(
    'Random Forest (tuned)',
    rf_search.best_estimator_,
    X_train,
    y_train,
    X_test,
    y_test,
    cv
)

{'model': 'Random Forest (tuned)', 'cv_roc_auc_mean': np.float64(0.902), 'test_accuracy': 0.898, 'test_precision': 0.938, 'test_recall': 0.563, 'test_f1': 0.704, 'test_roc_auc': 0.906}


### Compare tuned vs. baseline

In [7]:
rf_results = pd.DataFrame(results)

rf_results

,model,cv_roc_auc_mean,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,"Random Forest (baseline, raw)",0.898,0.897,0.939,0.558,0.700,0.900
1,"Random Forest (baseline, SMOTE-balanced)",0.978,0.884,0.790,0.632,0.702,0.893
2,Random Forest (tuned),0.902,0.898,0.938,0.563,0.704,0.906
